# Notebook 3 — Personas & Simulation des Migrations

> Profilage des clusters en personas marketing, split temporel et simulation de la détection de migrations.

**Projet** : BDD #7 Sephora × Albert School | Groupe 5 | Case 2  
**Seed** : 42 (reproductibilité garantie)

> **Migrations validées** : 2,609 migrations réelles (Juil–Sep 2025) — après filtre Oct–Déc 2025 et correction biais saisonnalité Q3. Sans correction : 13,794 migrations artificielles (×5.3).

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys, os

sys.path.append(os.path.join(os.getcwd(), '..'))  # accès au package src/

RANDOM_SEED = 42

---

## 3.1 Profilage des clusters

Calcul des KPIs par cluster et comparaison vs moyenne globale.


### KPIs par cluster

CA moyen, panier, fréquence, recency, discount rate, top marques, top axes.


In [2]:

import os, sys
sys.path.append(os.path.join(os.getcwd(), '..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.utils import DATA_PATH, MODELS_PATH
from src.clustering import load_model, preprocess
from src.personas import (profile_cluster, compute_delta_vs_global,
                           generate_persona_card, plot_radar_chart,
                           PERSONA_NAMES)

# Chargement modèle + features train (avec cluster)
model, scaler = load_model()
feat_train = pd.read_csv(os.path.join(DATA_PATH, "customer_features_train.csv"),
                          index_col="anonymized_card_code")

# Si colonne cluster absente, la recalculer
if "cluster" not in feat_train.columns:
    X, _, _ = preprocess(feat_train, scaler=scaler, fit=False)
    feat_train["cluster"] = model.predict(X)
    feat_train.to_csv(os.path.join(DATA_PATH, "customer_features_train.csv"))
    print("Clusters recalculés et sauvegardés")

# Profil des clusters
profile = profile_cluster(feat_train)
print(f"Clusters identifiés : {len(profile)}")
print("\n=== KPIs MOYENS PAR CLUSTER ===")
print(profile[["n_clients", "pct_clients", "monetary", "avg_basket",
               "frequency", "recency_days", "discount_ratio"]].round(2).to_string())


[load_model] Chargé : K=9, scaler fitted
Clusters identifiés : 9

=== KPIs MOYENS PAR CLUSTER ===
         n_clients  pct_clients  monetary  avg_basket  frequency  recency_days  discount_ratio
cluster                                                                                       
0             8619        21.04    114.26      100.96       1.15        180.01            0.25
1             6180        15.08    117.33       84.15       1.60        179.17            0.08
2             6072        14.82     26.80       22.36       1.21        182.91            0.06
3             1183         2.89    889.67      138.46       9.47        114.94            0.19
4             7369        17.99     44.94       40.21       1.14        183.56            0.04
5             3537         8.63    142.87       52.55       2.89        128.17            0.13
6             4255        10.39    298.20       79.27       4.43        129.08            0.14
7             1065         2.60    109.58      

### Delta vs moyenne globale

Obligatoire : chaque KPI comparé à la moyenne du dataset.


In [3]:

# Calcul des deltas vs moyenne globale
profile_with_delta = compute_delta_vs_global(profile)

print("=== DELTA vs MOYENNE GLOBALE (colonnes delta_*) ===")
delta_cols = [c for c in profile_with_delta.columns if c.startswith("delta_")]
print(profile_with_delta[delta_cols[:8]].round(1).to_string())

# Heatmap des deltas normalisés
kpi_base = ["monetary", "avg_basket", "frequency", "recency_days",
            "discount_ratio", "pct_estore", "icb_score"]
kpi_base = [c for c in kpi_base if c in profile.columns]

delta_vals = profile_with_delta[[f"delta_{c}_pct" for c in kpi_base
                                  if f"delta_{c}_pct" in profile_with_delta.columns]]

fig, ax = plt.subplots(figsize=(11, max(4, len(profile) * 0.7)))
im = ax.imshow(delta_vals.values, cmap="RdPu", aspect="auto", vmin=-100, vmax=100)
ax.set_xticks(range(len(delta_vals.columns)))
ax.set_yticks(range(len(delta_vals.index)))
ax.set_xticklabels([c.replace("delta_", "").replace("_pct", "") for c in delta_vals.columns],
                   rotation=30, ha="right")
ax.set_yticklabels([f"Cluster {c}" for c in delta_vals.index])

for i in range(len(delta_vals.index)):
    for j in range(len(delta_vals.columns)):
        val = delta_vals.iloc[i, j]
        sign = "+" if val >= 0 else ""
        color = "white" if abs(val) > 60 else "#1A1A1A"
        ax.text(j, i, f"{sign}{val:.0f}%", ha="center", va="center",
                fontsize=9, color=color, fontweight="bold")

plt.colorbar(im, ax=ax, label="Delta vs moyenne (%)")
ax.set_title("Delta KPIs vs Moyenne Globale par Cluster", fontweight="bold")
fig.tight_layout()
plt.savefig("../outputs/figures/3_delta_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


=== DELTA vs MOYENNE GLOBALE (colonnes delta_*) ===
         delta_recency_days_pct  delta_frequency_pct  delta_monetary_pct  delta_avg_basket_pct  delta_discount_ratio_pct  delta_pct_estore_pct  delta_brand_entropy_pct  delta_icb_score_pct
cluster                                                                                                                                                                                     
0                           9.0                -46.3               -18.1                  51.2                      96.9                   7.5                    -69.1                -70.7
1                           8.5                -25.1               -15.9                  26.0                     -35.5                 -75.0                     53.8                 62.8
2                          10.7                -43.5               -80.8                 -66.5                     -53.1                 -87.3                    -95.3                -84.7
3  

/var/folders/c7/2p6zt6l53dgdzbcb0r6hz2300000gn/T/ipykernel_36503/1567475919.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Comparaison avec RFM Sephora

Overlap entre nos clusters et les RFM_Segment_ID existants.


In [4]:

from src.personas import compare_with_sephora_rfm

if "RFM_Segment_ID" in feat_train.columns:
    cross = compare_with_sephora_rfm(feat_train)
    print("=== OVERLAP CLUSTERS × RFM_SEGMENT_ID SEPHORA ===")
    print("(% des clients d'un cluster appartenant à chaque RFM_Segment_ID)")
    print(cross.round(1).to_string())

    # Insight
    print("\n→ Nos clusters ML recoupent et enrichissent les segments RFM Sephora")
    print("  → Présenter comme 'affinement granulaire' de la segmentation existante")
else:
    print("RFM_Segment_ID absent des features — impossible de faire la comparaison")


=== OVERLAP CLUSTERS × RFM_SEGMENT_ID SEPHORA ===
(% des clients d'un cluster appartenant à chaque RFM_Segment_ID)
RFM_Segment_ID     1     2     3     4     5     6     7     8    9  99999  Total_clients
cluster                                                                                  
0                3.2  10.8   5.1  22.5  11.3  22.1   6.6  18.1  0.2    0.2           8619
1                4.3  15.7  10.6  19.9   9.0  16.6   8.9  14.7  0.3    0.1           6180
2                0.6   6.2  18.7   7.8   1.1  12.9  43.2   9.6  0.0    0.0           6072
3               83.5   7.5   0.5   2.0   2.5   0.4   0.3   3.2  0.0    0.0           1183
4                1.2   7.7  13.0  11.4   1.9  16.9  30.5  14.8  2.4    0.1           7369
5                7.0  31.8  26.1  15.4   3.1   8.2   6.4   1.9  0.2    0.0           3537
6               28.1  37.3   8.4  14.3   4.6   2.7   1.2   3.2  0.1    0.0           4255
7                3.6  25.4  30.9  13.3   1.3  12.3  10.6   2.4  0.1    0.1 

---

## 3.2 Création des fiches personas

Nommage et description narrative de chaque cluster.


### Nommage des personas

Nom évocateur pour le marketing + description en une phrase.


In [5]:

# Fiches personas complètes
global_stats = profile.select_dtypes(include=[np.number]).mean()

persona_cards = []
print("=== FICHES PERSONAS ===\n")
for cl in sorted(profile.index):
    row = profile.loc[cl]
    card = generate_persona_card(cl, row, global_stats, feat_train)
    persona_cards.append(card)

    print(f"{'='*60}")
    print(f"CLUSTER {cl} — {card['name']}")
    print(f"  📊 {card['n_clients']:,} clients ({card['pct_clients']:.1f}%)")
    print(f"  💰 CA moyen       : {card['kpis']['monetary_mean']:.0f} € "
          f"({card['deltas_vs_global']['monetary']})")
    print(f"  🛒 Panier moyen   : {card['kpis']['avg_basket']:.0f} € "
          f"({card['deltas_vs_global']['avg_basket']})")
    print(f"  📅 Fréquence      : {card['kpis']['frequency']:.1f} tickets "
          f"({card['deltas_vs_global']['frequency']})")
    print(f"  ⏱️  Recency        : {card['kpis']['recency_days']:.0f} jours")
    print(f"  🎯 ICB Score      : {card['kpis']['icb_score']:.0f}/100")
    print(f"  💎 CLV estimée    : {card['kpis']['clv_estimated']:.0f} €")
    print(f"\n  RECOMMANDATIONS MARKETING :")
    for r in card["recommendations"]:
        print(f"    {r}")
    print()


=== FICHES PERSONAS ===

CLUSTER 0 — L'Acheteuse Occasionnelle
  📊 8,619 clients (21.0%)
  💰 CA moyen       : 114 € (-47.4%)
  🛒 Panier moyen   : 101 € (+43.1%)
  📅 Fréquence      : 1.1 tickets (-62.4%)
  ⏱️  Recency        : 180 jours
  🎯 ICB Score      : 4/100
  💎 CLV estimée    : 1218 €

  RECOMMANDATIONS MARKETING :
    🏪  Animation en magasin : échantillons, beauty consultations exclusives
    💰  Offres flash ciblées avec countdown — sensible aux promotions
    ⏰  Campagne réactivation : 'Vous nous manquez' + code promo à durée limitée

CLUSTER 1 — L'Exploratrice Curieuse
  📊 6,180 clients (15.1%)
  💰 CA moyen       : 117 € (-46.0%)
  🛒 Panier moyen   : 84 € (+19.3%)
  📅 Fréquence      : 1.6 tickets (-47.6%)
  ⏱️  Recency        : 179 jours
  🎯 ICB Score      : 24/100
  💎 CLV estimée    : 1416 €

  RECOMMANDATIONS MARKETING :
    🏪  Animation en magasin : échantillons, beauty consultations exclusives
    👑  Programme Sephora Rouge VIP : accès early sale, lancements exclusifs
    ⏰

### Radar charts

Visualisation 8 axes : Budget, Fidélité, Diversité, Premium, Digital, Promo, Skincare, Fragrance.


In [6]:

# Radar charts pour tous les clusters
global_max = profile.select_dtypes(include=[np.number]).max().to_dict()

print("Génération des radar charts...")
radar_paths = []
for cl in sorted(profile.index):
    path = plot_radar_chart(profile.loc[cl], cl, global_max,
                            global_mean_row=global_stats)
    radar_paths.append(path)
    print(f"  Cluster {cl} ({PERSONA_NAMES.get(cl, '?')}) → {path}")

# Afficher tous les radars dans une grille
n_clusters = len(profile)
n_cols = min(3, n_clusters)
n_rows = (n_clusters + n_cols - 1) // n_cols

fig, axes_r = plt.subplots(n_rows, n_cols, figsize=(7 * n_cols, 7 * n_rows),
                            subplot_kw={"polar": True})
if n_clusters == 1:
    axes_r = [[axes_r]]
elif n_rows == 1:
    axes_r = [axes_r]

axes_flat = [ax for row in axes_r for ax in row]

# Masquer les axes en trop
for ax in axes_flat[n_clusters:]:
    ax.set_visible(False)

fig.suptitle("Radar Charts — Tous les Personas Sephora", fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout()
plt.savefig("../outputs/figures/3_radar_all_personas.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"\n✓ Radar charts sauvegardés dans outputs/figures/")


Génération des radar charts...
  Cluster 0 (L'Acheteuse Occasionnelle) → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_radar_persona_0.png
  Cluster 1 (L'Exploratrice Curieuse) → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_radar_persona_1.png


  Cluster 2 (La Petite Acheteuse) → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_radar_persona_2.png
  Cluster 3 (La VIP Beauty Addict) → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_radar_persona_3.png
  Cluster 4 (Le Client Dormant) → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_radar_persona_4.png


  Cluster 5 (La Régulière Classique) → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_radar_persona_5.png
  Cluster 6 (La Passionnée Premium) → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_radar_persona_6.png
  Cluster 7 (La Fidèle Discrète) → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_radar_persona_7.png


  Cluster 8 (La Régulière Engagée) → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_radar_persona_8.png



✓ Radar charts sauvegardés dans outputs/figures/


/var/folders/c7/2p6zt6l53dgdzbcb0r6hz2300000gn/T/ipykernel_36503/3424417662.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

## 3.3 Indice de Curiosité Beauté (ICB)

Score propriétaire 0–100 mesurant la propension à explorer.


### Calcul de l'ICB

Combinaison pondérée de brand_entropy, axe_diversity, new_brand_rate, channel_diversity.


In [7]:

from src.personas import compute_beauty_curiosity_index

# ICB par cluster
icb = compute_beauty_curiosity_index(feat_train)
feat_train["icb_score_computed"] = icb

icb_by_cluster = feat_train.groupby("cluster")["icb_score_computed"].mean()
print("=== ICB MOYEN PAR CLUSTER ===")
for cl in sorted(icb_by_cluster.index):
    name = PERSONA_NAMES.get(cl, f"Cluster {cl}")
    score = icb_by_cluster[cl]
    print(f"  Cluster {cl} ({name:35s}) : ICB = {score:5.1f}/100")

# Clients à fort ICB (> 70)
n_high_icb = (feat_train["icb_score_computed"] >= 70).sum()
print(f"\nClients ICB >= 70 : {n_high_icb:,} ({n_high_icb/len(feat_train)*100:.1f}%)")
print("→ Ces clients sont les 'Beauty Addicts' à fort potentiel de CLV")


=== ICB MOYEN PAR CLUSTER ===
  Cluster 0 (L'Acheteuse Occasionnelle          ) : ICB =   4.3/100
  Cluster 1 (L'Exploratrice Curieuse            ) : ICB =  23.9/100
  Cluster 2 (La Petite Acheteuse                ) : ICB =   2.2/100
  Cluster 3 (La VIP Beauty Addict               ) : ICB =  53.1/100
  Cluster 4 (Le Client Dormant                  ) : ICB =   3.3/100
  Cluster 5 (La Régulière Classique             ) : ICB =  16.3/100
  Cluster 6 (La Passionnée Premium              ) : ICB =  37.1/100
  Cluster 7 (La Fidèle Discrète                 ) : ICB =  23.1/100
  Cluster 8 (La Régulière Engagée               ) : ICB =  28.1/100

Clients ICB >= 70 : 49 (0.1%)
→ Ces clients sont les 'Beauty Addicts' à fort potentiel de CLV


### Corrélation ICB / CLV

Montrer que les clients à fort ICB ont un CA supérieur à la moyenne.


In [8]:

# Corrélation ICB / CA
fig, ax = plt.subplots(figsize=(9, 5))

sample = feat_train.sample(min(5000, len(feat_train)), random_state=42)
scatter = ax.scatter(sample["icb_score_computed"], sample["monetary"],
                     alpha=0.3, s=15, c=sample["cluster"],
                     cmap="RdPu", vmin=0, vmax=model.n_clusters - 1)

# Ligne de régression
mask_valid = ~(sample["icb_score_computed"].isna() | sample["monetary"].isna())
x_vals = sample.loc[mask_valid, "icb_score_computed"].values
y_vals_raw = sample.loc[mask_valid, "monetary"].values
y_vals = y_vals_raw.clip(0, np.percentile(y_vals_raw, 99))

if len(x_vals) > 10:
    coefs = np.polyfit(x_vals, y_vals, 1)
    x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
    ax.plot(x_line, np.polyval(coefs, x_line), color="#FF0066", lw=2, label="Tendance")

corr = feat_train["icb_score_computed"].corr(feat_train["monetary"])
ax.set_title(f"ICB Score vs CA total — corrélation r = {corr:.3f}", fontweight="bold")
ax.set_xlabel("ICB Score (0–100)")
ax.set_ylabel("CA total client (€)")
plt.colorbar(scatter, ax=ax, label="Cluster")
ax.legend()
fig.tight_layout()
plt.savefig("../outputs/figures/3_icb_vs_ca.png", dpi=150, bbox_inches="tight")
plt.show()

# Groupes ICB
icb_bins = pd.cut(feat_train["icb_score_computed"], bins=[0, 30, 60, 70, 100],
                   labels=["ICB 0-30", "ICB 30-60", "ICB 60-70", "ICB 70-100"])
print("CA moyen par tranche ICB :")
print(feat_train.groupby(icb_bins, observed=True)["monetary"].mean().round(0))


CA moyen par tranche ICB :
icb_score_computed
ICB 0-30        92.0
ICB 30-60      330.0
ICB 60-70      997.0
ICB 70-100    2175.0
Name: monetary, dtype: float64


/var/folders/c7/2p6zt6l53dgdzbcb0r6hz2300000gn/T/ipykernel_36503/3396280789.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---

## 3.4 Split temporel et simulation

Division Jan–Juin / Juil–Sep, replay des transactions.


### Initialisation du feature store

Construire {client_id: profil} à partir de features_train.csv.


In [9]:

# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 1/3 — CHARGEMENT ET NETTOYAGE ROBUSTE
# Autonome : recharge tout depuis le disque si les variables ont été perdues.
# ═══════════════════════════════════════════════════════════════════════════
import importlib, os, sys, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

sys.path.append(os.path.join(os.getcwd(), ".."))

# ── Rechargement des modules (évite les caches stale du kernel) ─────────────
import src.migration_detector as _md
import src.clustering        as _cl
import src.personas          as _prs
importlib.reload(_md)
importlib.reload(_cl)
importlib.reload(_prs)

from src.utils             import DATA_PATH, OUTPUTS_PATH, MODELS_PATH
from src.clustering        import load_model, preprocess
from src.migration_detector import build_feature_store

# ── Chargement modèle ────────────────────────────────────────────────────────
model, scaler = load_model()
print(f"✅ Modèle chargé : K={model.n_clusters} clusters")

# ── Normalisation des IDs (float scientifique → entier en string) ────────────
def normalize_id(raw):
    """'-8.17068E+18' ou -8170680000000000000 → '-8170680000000000000'"""
    try:
        return "{:.0f}".format(float(str(raw).strip()))
    except (ValueError, OverflowError):
        return str(raw).strip()

# ── Nettoyage et re-sauvegarde de transactions_test.csv ─────────────────────
txn_path = os.path.join(DATA_PATH, "transactions_test.csv")
print(f"\nChargement {txn_path} …")

txn_test_full = pd.read_csv(txn_path, dtype=str, low_memory=False)
n_raw = len(txn_test_full)
print(f"  → {n_raw:,} lignes brutes")

# Normaliser ID client
txn_test_full["anonymized_card_code"] = txn_test_full["anonymized_card_code"].apply(normalize_id)

# Normaliser ID ticket
txn_test_full["anonymized_Ticket_ID"] = txn_test_full["anonymized_Ticket_ID"].apply(normalize_id)

# Harmoniser la date : accepte MM/DD/YYYY ou YYYY-MM-DD
txn_test_full["transactionDate"] = pd.to_datetime(
    txn_test_full["transactionDate"], infer_datetime_format=True, errors="coerce"
)
n_nat = txn_test_full["transactionDate"].isna().sum()
txn_test_full = txn_test_full.dropna(subset=["transactionDate"])

# Sauvegarder au format MM/DD/YYYY (attendu par replay_transactions)
txn_test_full["transactionDate"] = txn_test_full["transactionDate"].dt.strftime("%m/%d/%Y")
txn_test_full.to_csv(txn_path, index=False)

print(f"  → {len(txn_test_full):,} lignes nettoyées ({n_nat} NaT supprimées)")
print(f"  → transactions_test.csv réécrit ✅")
print(f"  → Plage dates : {txn_test_full['transactionDate'].iloc[0]}  …  {txn_test_full['transactionDate'].iloc[-1]}")

# ── Feature store (IDs en string pour éviter la troncature float64) ──────────
print("\nConstruction du feature store …")
feature_store = build_feature_store(os.path.join(DATA_PATH, "customer_features_train.csv"))

# ── Clusters initiaux ────────────────────────────────────────────────────────
feat_str = pd.read_csv(
    os.path.join(DATA_PATH, "customer_features_train.csv"),
    dtype={"anonymized_card_code": str},
    usecols=["anonymized_card_code", "cluster"],
    low_memory=False
)
feat_str["anonymized_card_code"] = feat_str["anonymized_card_code"].apply(normalize_id)

initial_clusters = {
    row["anonymized_card_code"]: int(row["cluster"])
    for _, row in feat_str.iterrows()
}

# ── Vérification overlap ─────────────────────────────────────────────────────
fs_ids  = set(feature_store.keys())
ic_ids  = set(initial_clusters.keys())
txn_ids = set(txn_test_full["anonymized_card_code"].unique())

overlap_fs_ic  = len(fs_ids  & ic_ids)
overlap_fs_txn = len(fs_ids  & txn_ids)

print(f"\n{'─'*55}")
print(f"  Feature store    : {len(fs_ids):>8,} clients")
print(f"  Initial clusters : {len(ic_ids):>8,} clients")
print(f"  Clients test     : {len(txn_ids):>8,} clients")
print(f"  Overlap FS ↔ IC  : {overlap_fs_ic:>8,} clients  {'✅' if overlap_fs_ic > 0 else '❌'}")
print(f"  Overlap FS ↔ txn : {overlap_fs_txn:>8,} clients  {'✅' if overlap_fs_txn > 0 else '❌'}")
print(f"{'─'*55}")

assert overlap_fs_ic  > 0, "❌ 0 overlap feature_store ↔ initial_clusters"
assert overlap_fs_txn > 0, "❌ 0 overlap feature_store ↔ transactions_test"

# Distribution initiale des clusters
dist_init = pd.Series(initial_clusters).value_counts().sort_index()
print("\nDistribution initiale des clusters :")
for cl, n in dist_init.items():
    print(f"  Cluster {cl} : {n:,} clients ({n/len(initial_clusters)*100:.1f}%)")


[load_model] Chargé : K=9, scaler fitted
✅ Modèle chargé : K=9 clusters

Chargement /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/data/transactions_test.csv …


  → 218,967 lignes brutes


  → 218,967 lignes nettoyées (0 NaT supprimées)
  → transactions_test.csv réécrit ✅
  → Plage dates : 11/29/2025  …  12/14/2025

Construction du feature store …


[build_feature_store] 40,972 clients chargés dans le feature store



───────────────────────────────────────────────────────
  Feature store    :   40,972 clients
  Initial clusters :   40,972 clients
  Clients test     :   47,932 clients
  Overlap FS ↔ IC  :   40,972 clients  ✅
  Overlap FS ↔ txn :   24,435 clients  ✅
───────────────────────────────────────────────────────

Distribution initiale des clusters :
  Cluster 0 : 8,619 clients (21.0%)
  Cluster 1 : 6,180 clients (15.1%)
  Cluster 2 : 6,072 clients (14.8%)
  Cluster 3 : 1,183 clients (2.9%)
  Cluster 4 : 7,369 clients (18.0%)
  Cluster 5 : 3,537 clients (8.6%)
  Cluster 6 : 4,255 clients (10.4%)
  Cluster 7 : 1,065 clients (2.6%)
  Cluster 8 : 2,692 clients (6.6%)


### Replay Juil–Sep

Traiter les transactions chronologiquement, mettre à jour les profils.


In [10]:

# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 2/3 — LANCEMENT DU REPLAY
# ═══════════════════════════════════════════════════════════════════════════
assert "feature_store"    in dir(), "❌ Lancer d'abord la cellule de chargement (cell-21)"
assert "initial_clusters" in dir(), "❌ Lancer d'abord la cellule de chargement (cell-21)"
assert "model"            in dir(), "❌ Lancer d'abord la cellule de chargement (cell-21)"

import importlib
import src.migration_detector as _md
importlib.reload(_md)
from src.migration_detector import replay_transactions

txn_path = os.path.join(DATA_PATH, "transactions_test.csv")
print(f"Fichier source   : {txn_path}")
print(f"Taille fichier   : {os.path.getsize(txn_path)/1024**2:.1f} MB")
print(f"Clients en stock : {len(feature_store):,}")
print(f"Clusters initiaux: {len(initial_clusters):,}")
print()
print("Démarrage du replay …")
print("─" * 55)

migrations, final_clusters = replay_transactions(
    transactions_test_path = txn_path,
    feature_store          = feature_store,
    initial_clusters       = initial_clusters,
    model                  = model,
    scaler                 = scaler,
    verbose                = True,
)

print("─" * 55)
print(f"\n✅ Replay terminé :")
print(f"   Migrations détectées : {len(migrations):,}")
print(f"   Clients trackés      : {len(final_clusters):,}")

if len(migrations) == 0:
    print("\n⚠️  0 migration — relancer Kernel → Restart & Run All")
else:
    mig_preview = pd.DataFrame(migrations).head(5)
    print("\nAperçu des 5 premières migrations :")
    print(mig_preview[["client_id","date","from_cluster","to_cluster","direction"]].to_string(index=False))


Fichier source   : /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/data/transactions_test.csv
Taille fichier   : 49.8 MB
Clients en stock : 40,972
Clusters initiaux: 40,972

Démarrage du replay …
───────────────────────────────────────────────────────


[replay] 135,888 lignes hors période (> 2025-09-30) exclues


  [replay] 5,000/39,890 tickets (12.5%) — 305 migrations détectées


  [replay] 10,000/39,890 tickets (25.1%) — 620 migrations détectées


  [replay] 15,000/39,890 tickets (37.6%) — 920 migrations détectées


  [replay] 20,000/39,890 tickets (50.1%) — 1257 migrations détectées


  [replay] 25,000/39,890 tickets (62.7%) — 1586 migrations détectées


  [replay] 30,000/39,890 tickets (75.2%) — 1940 migrations détectées


  [replay] 35,000/39,890 tickets (87.7%) — 2248 migrations détectées



[replay] Terminé : 2609 migrations sur 39,890 tickets
───────────────────────────────────────────────────────

✅ Replay terminé :
   Migrations détectées : 2,609
   Clients trackés      : 50,734

Aperçu des 5 premières migrations :
           client_id       date  from_cluster  to_cluster direction
 6221350000000000000 2025-07-01             8           3   upgrade
-7486690000000000000 2025-07-01             6           3   upgrade
 2970870000000000000 2025-07-01             8           3   upgrade
-6032200000000000000 2025-07-01             6           3   upgrade
-8802440000000000000 2025-07-01             6           3   upgrade


### Détection des migrations

Comparer ancien et nouveau cluster pour chaque transaction.


In [11]:

# ── Analyse des migrations détectées ────────────────────────────────────────
import pandas as pd

mig_df = pd.DataFrame(migrations) if migrations else pd.DataFrame(
    columns=["client_id","txn_id","date","from_cluster","to_cluster","direction"]
)

if len(mig_df) > 0:
    mig_df["date"] = pd.to_datetime(mig_df["date"])

    print(f"{'═'*55}")
    print(f"  RÉSULTATS DU REPLAY — {len(mig_df):,} migrations")
    print(f"{'═'*55}")
    print(f"  Clients qui ont migré  : {mig_df['client_id'].nunique():,}")
    print(f"  Période couverte       : {mig_df['date'].min().date()} → {mig_df['date'].max().date()}")
    print()

    # Direction
    print("Direction des migrations :")
    dir_counts = mig_df["direction"].value_counts()
    for d, n in dir_counts.items():
        pct = n / len(mig_df) * 100
        print(f"  {d:12s} : {n:,} ({pct:.1f}%)")

    # Flux top 10
    print("\nFlux les plus fréquents (Top 10) :")
    flux = (mig_df.groupby(["from_cluster","to_cluster"])
                  .size()
                  .sort_values(ascending=False)
                  .head(10)
                  .reset_index(name="count"))
    for _, r in flux.iterrows():
        print(f"  Cluster {int(r['from_cluster'])} → Cluster {int(r['to_cluster'])} : {int(r['count']):,}")

    # Timeline mensuelle
    print("\nTimeline mensuelle des migrations :")
    monthly = mig_df.groupby(mig_df["date"].dt.to_period("M")).size()
    for period, n in monthly.items():
        print(f"  {period} : {n:,} migrations")

else:
    print("⚠️  Aucune migration détectée.")
    print("   → Vérifier que cell-21 affiche bien 'Overlap FS ↔ txn > 0'")
    print("   → Puis relancer Kernel → Restart & Run All")


═══════════════════════════════════════════════════════
  RÉSULTATS DU REPLAY — 2,609 migrations
═══════════════════════════════════════════════════════
  Clients qui ont migré  : 2,524
  Période couverte       : 2025-07-01 → 2025-09-30

Direction des migrations :
  downgrade    : 1,773 (68.0%)
  upgrade      : 836 (32.0%)

Flux les plus fréquents (Top 10) :
  Cluster 1 → Cluster 6 : 497
  Cluster 1 → Cluster 5 : 457
  Cluster 6 → Cluster 3 : 411
  Cluster 8 → Cluster 3 : 296
  Cluster 5 → Cluster 6 : 228
  Cluster 0 → Cluster 5 : 177
  Cluster 4 → Cluster 5 : 153
  Cluster 2 → Cluster 5 : 124
  Cluster 0 → Cluster 1 : 41
  Cluster 7 → Cluster 5 : 39

Timeline mensuelle des migrations :
  2025-07 : 924 migrations
  2025-08 : 831 migrations
  2025-09 : 854 migrations


---

## 3.5 Analyse des migrations

Matrice de transition et signaux précurseurs.

> **Périmètre** : 2,609 migrations réelles sur Juil–Sep 2025. Filtre `transactionDate <= 2025-09-30` appliqué dans `replay_transactions()`. Biais saisonnalité Q3 neutralisé via features corrigées.

### Matrice de migration

Tableau NxN des flux entre clusters sur la période Juil–Sep.

_Base : 2,609 migrations réelles (après correction Oct–Déc et saisonnalité Q3)._

In [12]:

# ═══════════════════════════════════════════════════════════════════════════
# CELLULE 3/3 — SAUVEGARDE COMPLÈTE DES OUTPUTS
# Garantit que le Notebook 4 et le Dashboard reçoivent les bonnes données.
# ═══════════════════════════════════════════════════════════════════════════
import importlib
import src.migration_detector as _md
import src.personas           as _prs
importlib.reload(_md)
importlib.reload(_prs)

from src.migration_detector import build_migration_matrix, save_migrations, plot_migration_heatmap
from src.personas            import save_personas, profile_cluster, compute_delta_vs_global, generate_persona_card

os.makedirs(os.path.join(OUTPUTS_PATH, "data"),    exist_ok=True)
os.makedirs(os.path.join(OUTPUTS_PATH, "figures"), exist_ok=True)

# ── 1. Matrice de migration ──────────────────────────────────────────────────
matrix = build_migration_matrix(migrations, model.n_clusters)

print("Matrice de migration (lignes = origine | colonnes = destination) :")
print(matrix.to_string())

# Taux de migration par cluster
if len(mig_df) > 0:
    print("\nTaux de migration par cluster :")
    for cl in range(model.n_clusters):
        n_init = sum(1 for v in initial_clusters.values() if v == cl)
        n_mig  = mig_df[mig_df["from_cluster"] == cl]["client_id"].nunique()
        if n_init > 0:
            print(f"  Cluster {cl} : {n_mig:,} / {n_init:,} clients ({n_mig/n_init*100:.1f}%)")

# ── 2. Sauvegarde migrations.csv + migration_matrix.csv ─────────────────────
paths = save_migrations(migrations, matrix)
print(f"\n✅ migrations.csv        → {paths['migrations']}")
if "matrix" in paths:
    print(f"✅ migration_matrix.csv  → {paths['matrix']}")

# Vérification taille fichier
mig_csv_path = paths["migrations"]
n_mig_lines  = sum(1 for _ in open(mig_csv_path)) - 1
print(f"   ({n_mig_lines:,} lignes dans migrations.csv)")

# ── 3. Heatmap migration ────────────────────────────────────────────────────
heatmap_path = plot_migration_heatmap(matrix)
print(f"✅ heatmap               → {heatmap_path}")

# ── 4. Personas — profil + delta + fiches ───────────────────────────────────
feat_train = pd.read_csv(
    os.path.join(DATA_PATH, "customer_features_train.csv"),
    dtype={"anonymized_card_code": str},
    index_col="anonymized_card_code"
)

profile          = profile_cluster(feat_train)
profile_with_delta = compute_delta_vs_global(profile)
global_stats     = profile.select_dtypes(include=[np.number]).mean()

persona_cards = []
for cl in sorted(profile.index):
    card = generate_persona_card(cl, profile.loc[cl], global_stats, feat_train)
    persona_cards.append(card)

persona_paths = save_personas(profile_with_delta, persona_cards)
print(f"✅ personas_profiles.csv → {persona_paths.get('profiles', '?')}")
print(f"✅ persona_cards.json    → {persona_paths.get('cards', '?')}")

# ── 5. KPIs business ────────────────────────────────────────────────────────
kpi_cols = ["n_clients","pct_clients","monetary","avg_basket","frequency",
            "recency_days","discount_ratio","pct_estore","icb_score","clv_estimated"]
kpi_cols = [c for c in kpi_cols if c in profile_with_delta.columns]
kpi_path = os.path.join(OUTPUTS_PATH, "data", "business_kpis.csv")
profile_with_delta[kpi_cols].to_csv(kpi_path)
print(f"✅ business_kpis.csv     → {kpi_path}")

print(f"\n{'═'*55}")
print(f"  PIPELINE COMPLET — tous les outputs sauvegardés")
print(f"  Lancer : streamlit run app/dashboard.py")
print(f"{'═'*55}")


Matrice de migration (lignes = origine | colonnes = destination) :
           Cluster 0  Cluster 1  Cluster 2  Cluster 3  Cluster 4  Cluster 5  Cluster 6  Cluster 7  Cluster 8
Cluster 0          0         41         11          2         10        177         24          0          3
Cluster 1          1          0          1          1          0        457        497         21          0
Cluster 2          2          9          0          0          0        124          0          0          5
Cluster 3          0          0          0          0          0          0          0          0          0
Cluster 4          4         35          7          0          0        153          1          0          4
Cluster 5          0          0          0          5          0          0        228         13          0
Cluster 6          0          0          0        411          0          0          0          0          0
Cluster 7          0          0          0          1        

✅ heatmap               → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_migration_heatmap.png
[save_personas] → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/data/personas_profiles.csv
[save_personas] → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/data/persona_cards.json
✅ personas_profiles.csv → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/data/personas_profiles.csv
✅ persona_cards.json    → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/data/persona_cards.json
✅ business_kpis.csv     → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/data/business_kpis.csv

═══════════════════════════════════════════════════════
  PIPELINE COMPLET — tous les outputs sauvegardés
  Lancer : streamlit run app/dashboard.py
═══════════════════════════════════════════════════════


### Sankey diagram

Visualisation des flux de migration — figure principale de la présentation.


In [13]:

# ── Sankey diagram (interactif HTML) ────────────────────────────────────────
import importlib
import src.migration_detector as _md
importlib.reload(_md)
from src.migration_detector import plot_sankey_migrations

if len(migrations) > 0:
    sankey_path = plot_sankey_migrations(migrations, model.n_clusters)
    print(f"✅ Sankey diagram → {sankey_path}")
    print("   Ouvrir ce fichier HTML dans un navigateur pour voir le diagramme interactif.")
else:
    print("⚠️  Pas de migrations — Sankey non généré")


[plot_sankey] plotly non disponible — fallback heatmap uniquement


✅ Sankey diagram → /Users/eliottmusy/Desktop/Sephora /sephora-segmentation-groupe5/outputs/figures/3_migration_heatmap.png
   Ouvrir ce fichier HTML dans un navigateur pour voir le diagramme interactif.


### Signaux précurseurs

Quel type d'achat déclenche une migration ? Analyse par axe et marque.


In [14]:

# ── Signaux précurseurs de migration ────────────────────────────────────────
if len(mig_df) > 0:
    txn_test_path = os.path.join(DATA_PATH, "transactions_test.csv")
    txn_test = pd.read_csv(txn_test_path, dtype={"anonymized_card_code": str}, low_memory=False)
    txn_test["transactionDate"] = pd.to_datetime(
        txn_test["transactionDate"], format="%m/%d/%Y", errors="coerce"
    )

    # Appliquer le même normalize_id que dans cell-21
    def normalize_id(raw):
        try:
            return "{:.0f}".format(float(str(raw).strip()))
        except (ValueError, OverflowError):
            return str(raw).strip()

    txn_test["anonymized_card_code"] = txn_test["anonymized_card_code"].apply(normalize_id)

    migrant_ids = set(mig_df["client_id"].unique())
    trigger_txns = txn_test[txn_test["anonymized_card_code"].isin(migrant_ids)]

    print(f"Transactions des {len(migrant_ids):,} clients ayant migré : {len(trigger_txns):,} lignes")
    print()

    if "Axe_Desc" in trigger_txns.columns:
        print("=== AXES D'ACHAT DES CLIENTS MIGRÉS ===")
        axes = trigger_txns["Axe_Desc"].value_counts(normalize=True).mul(100)
        for axe, pct in axes.items():
            print(f"  {axe:<25s} : {pct:.1f}%")

    if "brand" in trigger_txns.columns:
        print("\n=== TOP 15 MARQUES (clients migrés) ===")
        brands = trigger_txns["brand"].value_counts().head(15)
        for brand, n in brands.items():
            print(f"  {brand:<30s} : {n:,}")

    if "Market_Desc" in trigger_txns.columns:
        print("\n=== MARKET MIX (clients migrés) ===")
        market = trigger_txns["Market_Desc"].value_counts(normalize=True).mul(100)
        for m, pct in market.items():
            print(f"  {m:<20s} : {pct:.1f}%")

    print("\n→ Insight : ces signaux permettent de déclencher une action CRM préventive")
    print("  avant que la migration ne soit confirmée.")
else:
    print("⚠️  Pas de migrations — analyse des signaux précurseurs ignorée.")
    print("   Relancer les cellules 21 → 23 → 28 pour générer les migrations.")


Transactions des 2,524 clients ayant migré : 29,434 lignes

=== AXES D'ACHAT DES CLIENTS MIGRÉS ===
  MAKE UP                   : 49.1%
  SKINCARE                  : 29.1%
  FRAGRANCE                 : 11.5%
  HAIRCARE                  : 5.0%
  NOT SPECIFIED             : 3.4%
  OTHERS                    : 1.9%

=== TOP 15 MARQUES (clients migrés) ===
  SEPHORA                        : 9,443
  DIOR                           : 1,220
  HUDA BEAUTY                    : 1,146
  BENEFIT                        : 1,131
  SOL DE JANEIRO                 : 807
  CHARLOTTE TILBURY              : 737
  RARE BEAUTY                    : 555
  FENTY                          : 552
  THE ORDINARY                   : 545
  BYOMA                          : 530
  LANCOME                        : 475
  TOO FACED                      : 473
  GUERLAIN                       : 471
  YVES ST LAURENT                : 460
  ERBORIAN                       : 459

=== MARKET MIX (clients migrés) ===
  EXCLUSIVE     